# Exercise 02 — SOLUTION: Optimized Synaptic Input

Try the stub first! Only open this after attempting [ex02_stub.ipynb](ex02_stub.ipynb).

---

In [ ]:
!nvidia-smi

In [ ]:
%%writefile synaptic_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e)); exit(1);}} while(0)
#define TILE 16

__constant__ float c_g_syn;  // Challenge: synaptic conductance

// Naive: uncoalesced — W[i*N+j] has stride N between threads in a warp
__global__ void matvec_naive(const float* W, const float* s, float* I, int N) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= N) return;
    float sum = 0;
    for (int i = 0; i < N; i++) sum += W[i*N+j] * s[i];
    I[j] = sum;
}

// Coalesced: thread j reads Wt[j*N+i] — stride 1 between warp threads ✓
__global__ void matvec_coalesced(const float* Wt, const float* s, float* I, int N) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= N) return;
    float sum = 0;
    for (int i = 0; i < N; i++) sum += Wt[j*N+i] * s[i];  // stride-1 access
    I[j] = sum;
}

// Tiled: shared memory for s[]; Wt row still coalesced
__global__ void matvec_tiled(const float* Wt, const float* s, float* I, int N) {
    __shared__ float s_tile[TILE];

    int j = blockIdx.x * blockDim.x + threadIdx.x;
    float sum = 0;

    for (int t = 0; t < (N + TILE - 1) / TILE; t++) {
        // Collaboratively load s into shared memory
        if (threadIdx.x < TILE) {
            int idx = t * TILE + threadIdx.x;
            s_tile[threadIdx.x] = (idx < N) ? s[idx] : 0.0f;
        }
        __syncthreads();

        // Each thread processes TILE elements of its row using shared s
        if (j < N) {
            for (int k = 0; k < TILE; k++) {
                int i = t * TILE + k;
                if (i < N) sum += Wt[j*N+i] * s_tile[k];  // shared mem read
            }
        }
        __syncthreads();
    }

    if (j < N) I[j] = c_g_syn * sum;  // Challenge: scale by conductance
}

void matvec_cpu(const float* W, const float* s, float* I, int N) {
    for (int j = 0; j < N; j++) {
        float sum = 0;
        for (int i = 0; i < N; i++) sum += W[i*N+j] * s[i];
        I[j] = sum;
    }
}

float max_err(const float* a, const float* b, int n) {
    float e = 0;
    for (int i = 0; i < n; i++) { float d=fabsf(a[i]-b[i]); if(d>e)e=d; }
    return e;
}

int main(int argc, char** argv) {
    int N = (argc > 1) ? atoi(argv[1]) : 1024;
    size_t mb = (size_t)N*N*sizeof(float), vb = N*sizeof(float);

    // Challenge: set constant memory
    float g_syn = 0.5f;
    CUDA_CHECK(cudaMemcpyToSymbol(c_g_syn, &g_syn, sizeof(float)));

    float *hW=(float*)malloc(mb), *hWt=(float*)malloc(mb);
    float *hs=(float*)malloc(vb), *hI=(float*)malloc(vb), *href=(float*)malloc(vb);
    srand(42);
    for (int i=0;i<N*N;i++) hW[i]=(float)rand()/RAND_MAX*0.01f;
    for (int j=0;j<N;j++) hs[j]=(float)rand()/RAND_MAX;
    for (int i=0;i<N;i++) for (int j=0;j<N;j++) hWt[j*N+i]=hW[i*N+j];

    matvec_cpu(hW, hs, href, N);
    for (int j=0;j<N;j++) href[j] *= g_syn;  // scale reference too

    float *dW,*dWt,*ds,*dI;
    CUDA_CHECK(cudaMalloc(&dW,mb)); CUDA_CHECK(cudaMalloc(&dWt,mb));
    CUDA_CHECK(cudaMalloc(&ds,vb)); CUDA_CHECK(cudaMalloc(&dI,vb));
    CUDA_CHECK(cudaMemcpy(dW,hW,mb,cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dWt,hWt,mb,cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(ds,hs,vb,cudaMemcpyHostToDevice));

    int thr=256, blk=(N+thr-1)/thr;
    cudaEvent_t t0,t1; float ms;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));

    printf("N=%d\n",N);
    printf("%-18s  %-10s  %-10s  %-8s\n","Kernel","ms","GB/s","Status");

    #define BENCH(lbl,call,check_ptr) \
        CUDA_CHECK(cudaEventRecord(t0)); call; \
        CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1)); \
        CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1)); \
        CUDA_CHECK(cudaMemcpy(hI,dI,vb,cudaMemcpyDeviceToHost)); \
        printf("%-18s  %-10.3f  %-10.1f  %s\n",lbl,ms, \
               (mb+vb*2.0)/(ms*1e-3)/1e9, max_err(hI,check_ptr,N)<1e-3f?"PASS":"FAIL");

    // For naive, reference without g_syn scaling
    float *href_raw=(float*)malloc(vb);
    matvec_cpu(hW,hs,href_raw,N);
    BENCH("Naive",     matvec_naive<<<blk,thr>>>(dW,ds,dI,N), href_raw)
    BENCH("Coalesced", matvec_coalesced<<<blk,thr>>>(dWt,ds,dI,N), href_raw)
    BENCH("Tiled+const",matvec_tiled<<<blk,thr>>>(dWt,ds,dI,N), href)

    free(href_raw);
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(dW);cudaFree(dWt);cudaFree(ds);cudaFree(dI);
    free(hW);free(hWt);free(hs);free(hI);free(href);
    return 0;
}

In [ ]:
!nvcc -O2 -o synaptic_sol synaptic_sol.cu -lm && ./synaptic_sol 1024

In [ ]:
# Sweep N and plot
import subprocess, numpy as np, matplotlib.pyplot as plt

Ns = [128, 256, 512, 1024]
results = {}

for N in Ns:
    out = subprocess.run(['./synaptic_sol', str(N)], capture_output=True, text=True).stdout
    print(out)

print("Optimization insight: tiled kernel achieves best bandwidth due to shared memory caching of s[].")